# Model Evaluation and Validation

Training a machine learning model is only half the work. The critical question is: **how well does it perform on data it has never seen?** A model that memorizes training examples but fails on new data is useless in production. Evaluation and validation provide the discipline to measure true performance, compare models fairly, detect overfitting, and build confidence that a model will work in the real world.

This notebook covers the full evaluation toolkit: how to split data properly, which metrics to use for regression and classification, how to handle imbalanced classes, cross-validation strategies, learning curves, hyperparameter tuning, and the pitfalls that produce misleadingly optimistic results.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, make_regression, load_breast_cancer
from sklearn.model_selection import (train_test_split, cross_val_score, KFold,
                                      StratifiedKFold, learning_curve, GridSearchCV)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score,
                              accuracy_score, precision_score, recall_score, f1_score,
                              confusion_matrix, ConfusionMatrixDisplay,
                              classification_report, roc_curve, roc_auc_score,
                              precision_recall_curve, average_precision_score)

np.set_printoptions(precision=4, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")
np.random.seed(42)

---

## 1. Why Evaluation Matters

A model's **training performance** measures how well it fits the data it learned from. **Test performance** measures how well it generalizes to unseen data. The gap between them reveals overfitting.

Consider a fraud detection model with 99.9% training accuracy. Impressive — until test accuracy drops to 60%. The model memorized training fraud patterns but cannot detect new fraud schemes. Without proper evaluation, this failure would only surface in production, where it costs real money.

Good evaluation practice ensures:

- Performance estimates reflect **real-world** behavior.
- Models are compared **fairly** on the same data splits.
- **Overfitting** is detected before deployment.
- The right **metric** is chosen for the business problem.
- **Hyperparameters** are tuned without leaking test information.

---

## 2. Data Splitting Strategies

### 2.1 Train-Test Split

The simplest approach: divide data into a **training set** (used to fit the model) and a **test set** (used only for final evaluation). A common ratio is 80% train / 20% test.

The test set must remain **untouched** during all development — model selection, feature engineering, hyperparameter tuning. Touching the test set during development invalidates it as an unbiased performance estimate.

### 2.2 Train-Validation-Test Split

A three-way split adds a **validation set**:

- **Training set** — fit model parameters.
- **Validation set** — compare models, tune hyperparameters, decide when to stop training.
- **Test set** — final unbiased evaluation, used once at the end.

Typical split: 60% train / 20% validation / 20% test.

### 2.3 Stratified Splitting

For classification with imbalanced classes, **stratified splitting** preserves the class ratio in each split. If 5% of data is positive class, each split contains approximately 5% positives. Without stratification, a small test set might contain zero positive examples by chance.

In [ ]:
# Train-validation-test split
X, y = make_classification(n_samples=1000, n_features=10, weights=[0.9, 0.1], random_state=42)

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

print(f"Total samples: {len(y)}")
print(f"Train: {len(y_train)} ({y_train.mean():.1%} positive)")
print(f"Val:   {len(y_val)} ({y_val.mean():.1%} positive)")
print(f"Test:  {len(y_test)} ({y_test.mean():.1%} positive)")

---

## 3. Cross-Validation

A single train-test split gives a performance estimate that depends on which samples ended up in the test set. **Cross-validation** provides a more robust estimate by using every sample as test data exactly once.

### 3.1 K-Fold Cross-Validation

Divide data into $K$ equal folds. For each fold $i$:

1. Use fold $i$ as the test set.
2. Train on the remaining $K-1$ folds.
3. Record the test score.

The final score is the mean (and standard deviation) across all $K$ folds. $K=5$ or $K=10$ are common choices.

### 3.2 Stratified K-Fold

Preserves class proportions in each fold. Essential for imbalanced classification.

### 3.3 Leave-One-Out (LOO)

$K = n$ — each sample is its own test set. Unbiased but computationally expensive and high variance.

### 3.4 When to Use Cross-Validation

- Model selection and hyperparameter tuning.
- When data is limited and a single split would waste samples.
- To estimate performance variance (via standard deviation across folds).

After cross-validation guides your choices, evaluate the final model once on the held-out test set.

In [ ]:
# K-Fold cross-validation comparison
cancer = load_breast_cancer()
X_c, y_c = cancer.data, cancer.target

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
}

print(f"{'Model':25s} {'CV Mean':>10s} {'CV Std':>10s}")
print("-" * 47)
for name, model in models.items():
    scores = cross_val_score(model, X_c, y_c, cv=StratifiedKFold(5, shuffle=True, random_state=42),
                              scoring="accuracy")
    print(f"{name:25s} {scores.mean():10.4f} {scores.std():10.4f}")

---

## 4. Regression Metrics

Regression models predict continuous values. The right metric depends on whether large errors are especially costly and whether the target scale matters.

### 4.1 Mean Squared Error (MSE)

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

Penalizes large errors heavily (squaring). Sensitive to outliers. Not in the same units as the target.

### 4.2 Root Mean Squared Error (RMSE)

$$\text{RMSE} = \sqrt{\text{MSE}}$$

Same units as the target. More interpretable than MSE.

### 4.3 Mean Absolute Error (MAE)

$$\text{MAE} = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$$

Robust to outliers. Treats all errors linearly.

### 4.4 R² (Coefficient of Determination)

$$R^2 = 1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2}$$

Proportion of variance explained. $R^2 = 1$ is perfect; $R^2 = 0$ means the model is no better than predicting the mean; $R^2 < 0$ means the model is worse than the mean.

### 4.5 Mean Absolute Percentage Error (MAPE)

$$\text{MAPE} = \frac{100}{n} \sum_{i=1}^{n} \left|\frac{y_i - \hat{y}_i}{y_i}\right|$$

Scale-independent percentage error. Problematic when targets are near zero.

In [ ]:
# Regression metrics demonstration
y_true = np.array([3.0, -0.5, 2.0, 7.0, 4.2])
y_pred = np.array([2.5, 0.0, 2.1, 7.8, 3.9])

print(f"MSE:  {mean_squared_error(y_true, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_true, y_pred)):.4f}")
print(f"MAE:  {mean_absolute_error(y_true, y_pred):.4f}")
print(f"R²:   {r2_score(y_true, y_pred):.4f}")

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(y_true, y_pred, s=80, zorder=3)
lims = [min(y_true.min(), y_pred.min()) - 1, max(y_true.max(), y_pred.max()) + 1]
ax.plot(lims, lims, "r--", label="Perfect prediction")
ax.set_xlabel("True value")
ax.set_ylabel("Predicted value")
ax.set_title("Predicted vs True")
ax.legend()
plt.show()

---

## 5. Classification Metrics

Classification metrics are more nuanced than regression because the cost of different error types varies by problem.

### 5.1 Accuracy

$$\text{Accuracy} = \frac{\text{Correct predictions}}{\text{Total predictions}}$$

Simple and intuitive, but **misleading with imbalanced classes**. A model predicting "no fraud" for every transaction achieves 99% accuracy when only 1% are fraudulent — while catching zero fraud.

### 5.2 Precision

$$\text{Precision} = \frac{TP}{TP + FP}$$

Of all positive predictions, how many were correct? High precision means few false alarms. Important when false positives are costly (spam filter marking legitimate email as spam).

### 5.3 Recall (Sensitivity, True Positive Rate)

$$\text{Recall} = \frac{TP}{TP + FN}$$

Of all actual positives, how many did we catch? High recall means few missed cases. Important when false negatives are costly (cancer screening missing a tumor).

### 5.4 Specificity (True Negative Rate)

$$\text{Specificity} = \frac{TN}{TN + FP}$$

Of all actual negatives, how many did we correctly identify?

### 5.5 F1 Score

$$F_1 = 2 \cdot \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$

Harmonic mean of precision and recall. Balances both when you need a single number. $F_\beta$ generalizes this with different weighting.

---

## 6. The Confusion Matrix

The confusion matrix is the foundation of classification evaluation. For binary classification:

```
                    Predicted
                 Pos      Neg
Actual  Pos      TP       FN
        Neg      FP       TN
```

- **True Positive (TP)** — correctly predicted positive.
- **False Positive (FP)** — incorrectly predicted positive (Type I error).
- **False Negative (FN)** — incorrectly predicted negative (Type II error).
- **True Negative (TN)** — correctly predicted negative.

All classification metrics derive from these four counts.

In [ ]:
# Confusion matrix and derived metrics
y_true_bin = np.array([1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0])
y_pred_bin = np.array([1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0])

cm = confusion_matrix(y_true_bin, y_pred_bin)
tn, fp, fn, tp = cm.ravel()

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=["Neg", "Pos"]).plot(ax=ax, cmap="Blues")
plt.title("Confusion Matrix")
plt.show()

print(f"TP={tp}, FP={fp}, FN={fn}, TN={tn}")
print(f"Accuracy:    {(tp+tn)/(tp+fp+fn+tn):.3f}")
print(f"Precision:   {tp/(tp+fp):.3f}")
print(f"Recall:      {tp/(tp+fn):.3f}")
print(f"Specificity: {tn/(tn+fp):.3f}")
print(f"F1 Score:    {2*tp/(2*tp+fp+fn):.3f}")

---

## 7. ROC Curve and AUC

Many classifiers output **probabilities** rather than hard labels. The classification threshold (default 0.5) can be adjusted to trade precision for recall.

The **ROC curve** (Receiver Operating Characteristic) plots **True Positive Rate** (recall) vs **False Positive Rate** ($1 - \text{specificity}$) at every threshold:

$$\text{TPR} = \frac{TP}{TP + FN}, \quad \text{FPR} = \frac{FP}{FP + TN}$$

**AUC** (Area Under the Curve) summarizes ROC performance in a single number:

- AUC = 1.0 — perfect classifier.
- AUC = 0.5 — random guessing (diagonal line).
- AUC < 0.5 — worse than random.

AUC measures the model's ability to **rank** positives above negatives, independent of the chosen threshold. It is threshold-invariant and useful for comparing models.

**Limitation:** ROC-AUC can be optimistic with highly imbalanced data because a low FPR is easy to achieve when negatives dominate.

In [ ]:
# ROC curves for multiple models
X_tr, X_te, y_tr, y_te = train_test_split(cancer.data, cancer.target, test_size=0.3, random_state=42)
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

plt.figure(figsize=(8, 6))
for name, model in [("Logistic Regression", LogisticRegression(max_iter=1000)),
                     ("Random Forest", RandomForestClassifier(n_estimators=100, random_state=42))]:
    model.fit(X_tr_s, y_tr)
    y_prob = model.predict_proba(X_te_s)[:, 1]
    fpr, tpr, _ = roc_curve(y_te, y_prob)
    auc = roc_auc_score(y_te, y_prob)
    plt.plot(fpr, tpr, linewidth=2, label=f"{name} (AUC={auc:.3f})")

plt.plot([0, 1], [0, 1], "k--", label="Random (AUC=0.5)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves")
plt.legend()
plt.show()

---

## 8. Precision-Recall Curve

For **imbalanced datasets**, the Precision-Recall (PR) curve is often more informative than ROC.

The PR curve plots precision vs recall at every threshold. **Average Precision (AP)** summarizes the area under the PR curve.

- A model with high recall but low precision catches most positives but generates many false alarms.
- A model with high precision but low recall is conservative — few false alarms but misses many positives.

When the positive class is rare (fraud, disease, defect), PR curves give a clearer picture of practical performance than ROC curves.

In [ ]:
# PR curve on imbalanced data
X_imb, y_imb = make_classification(n_samples=2000, n_features=10, weights=[0.95, 0.05],
                                    random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_imb, y_imb, test_size=0.3, stratify=y_imb, random_state=42)

model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_tr, y_tr)
y_prob = model.predict_proba(X_te)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_te, y_prob)
ap = average_precision_score(y_te, y_prob)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(recall, precision, "b-", linewidth=2)
axes[0].set_xlabel("Recall")
axes[0].set_ylabel("Precision")
axes[0].set_title(f"PR Curve (AP={ap:.3f})")

fpr, tpr, _ = roc_curve(y_te, y_prob)
auc = roc_auc_score(y_te, y_prob)
axes[1].plot(fpr, tpr, "r-", linewidth=2)
axes[1].plot([0, 1], [0, 1], "k--")
axes[1].set_xlabel("FPR")
axes[1].set_ylabel("TPR")
axes[1].set_title(f"ROC Curve (AUC={auc:.3f})")

plt.tight_layout()
plt.show()
print(f"Class balance: {y_te.mean():.1%} positive")
print(f"ROC-AUC looks good ({auc:.3f}) but check if PR curve tells a different story")

---

## 9. Handling Class Imbalance

Real-world classification problems are often imbalanced: fraud (0.1%), disease (5%), defects (2%). Standard accuracy is misleading. Strategies:

### 9.1 Choose the Right Metric

Use precision, recall, F1, PR-AUC instead of accuracy. Define which error type is costlier and optimize accordingly.

### 9.2 Class Weighting

Penalize misclassification of the minority class more heavily. Many algorithms support `class_weight='balanced'`, which sets weights inversely proportional to class frequency.

### 9.3 Resampling

- **Oversampling** — duplicate or synthetically generate minority class samples (SMOTE).
- **Undersampling** — reduce majority class samples.
- Apply resampling only to the training set, never to the test set.

### 9.4 Threshold Tuning

Instead of the default 0.5 threshold, choose a threshold that maximizes F1 or meets a business constraint (e.g., recall ≥ 0.95).

In [ ]:
# Effect of class weighting on imbalanced data
for name, clf in [("No weighting", LogisticRegression(max_iter=1000)),
                   ("Balanced weights", LogisticRegression(max_iter=1000, class_weight="balanced"))]:
    clf.fit(X_tr, y_tr)
    y_p = clf.predict(X_te)
    print(f"\n{name}:")
    print(f"  Accuracy:  {accuracy_score(y_te, y_p):.4f}")
    print(f"  Precision: {precision_score(y_te, y_p, zero_division=0):.4f}")
    print(f"  Recall:    {recall_score(y_te, y_p, zero_division=0):.4f}")
    print(f"  F1:        {f1_score(y_te, y_p, zero_division=0):.4f}")

---

## 10. Learning Curves

Learning curves plot training and validation performance as a function of training set size. They diagnose whether more data will help and whether the model is overfitting or underfitting.

**High bias (underfitting):** Both training and validation scores are low and converge together. The model is too simple. Solution: more complex model, more features.

**High variance (overfitting):** Training score is high, validation score is significantly lower, and there is a large gap. The model memorizes training data. Solution: more data, regularization, simpler model.

**Good fit:** Training and validation scores are close and both reasonably high. Adding more data may still help if the curves have not converged.

In [ ]:
# Learning curves: underfitting vs overfitting
X_lc, y_lc = make_classification(n_samples=500, n_features=10, n_informative=5, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, model, title in zip(axes,
    [LogisticRegression(max_iter=1000), DecisionTreeClassifier(max_depth=None, random_state=42)],
    ["Logistic Regression (may underfit)", "Deep Decision Tree (may overfit)"]):
    train_sizes, train_scores, val_scores = learning_curve(
        model, X_lc, y_lc, cv=5, train_sizes=np.linspace(0.1, 1.0, 8), scoring="accuracy")
    ax.plot(train_sizes, train_scores.mean(axis=1), "o-", label="Train", linewidth=2)
    ax.fill_between(train_sizes, train_scores.mean(axis=1) - train_scores.std(axis=1),
                     train_scores.mean(axis=1) + train_scores.std(axis=1), alpha=0.1)
    ax.plot(train_sizes, val_scores.mean(axis=1), "o-", label="Validation", linewidth=2)
    ax.fill_between(train_sizes, val_scores.mean(axis=1) - val_scores.std(axis=1),
                     val_scores.mean(axis=1) + val_scores.std(axis=1), alpha=0.1)
    ax.set_xlabel("Training set size")
    ax.set_ylabel("Accuracy")
    ax.set_title(title)
    ax.legend()

plt.tight_layout()
plt.show()

---

## 11. Hyperparameter Tuning

Hyperparameters are settings chosen before training (learning rate, tree depth, number of neighbors) — not learned from data. Tuning them requires systematic search guided by validation performance.

### 11.1 Grid Search

Define a grid of hyperparameter combinations and evaluate each using cross-validation. Exhaustive but expensive — the search space grows exponentially.

### 11.2 Random Search

Sample random combinations from the hyperparameter space. Often finds good solutions faster than grid search, especially when only a few hyperparameters matter.

### 11.3 Bayesian Optimization

Build a probabilistic model of the objective function and use it to choose the next hyperparameters to try. More sample-efficient than random search.

**Critical rule:** Hyperparameter tuning must use cross-validation on the training set (or a validation set). Never tune hyperparameters using the test set.

In [ ]:
# Grid search with cross-validation
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier())
])

param_grid = {
    "knn__n_neighbors": [3, 5, 7, 9, 11, 15],
    "knn__weights": ["uniform", "distance"],
}

grid = GridSearchCV(pipe, param_grid, cv=5, scoring="accuracy", return_train_score=True)
grid.fit(cancer.data, cancer.target)

print(f"Best parameters: {grid.best_params_}")
print(f"Best CV accuracy: {grid.best_score_:.4f}")

results = pd.DataFrame(grid.cv_results_)
pivot = results.pivot_table(values="mean_test_score", index="param_knn__n_neighbors",
                             columns="param_knn__weights")
print("\nCV accuracy by hyperparameters:")
print(pivot.round(4).to_string())

---

## 12. Avoiding Evaluation Pitfalls

### 12.1 Data Leakage

Information from the test set leaking into training produces optimistically biased evaluation. Common sources:

- Fitting scalers, imputers, or feature selectors on the full dataset before splitting.
- Using target variable statistics for feature engineering before splitting.
- Including future information in time-series predictions.
- Duplicate or near-duplicate samples across train and test sets.

### 12.2 Multiple Testing

Trying many models and reporting only the best test score inflates performance estimates. If you try 20 models, one will look good by chance. Use cross-validation for model selection; reserve the test set for a single final evaluation.

### 12.3 Non-Representative Test Data

The test set must reflect the distribution the model will face in production. A model trained on historical data and tested on the same period may fail when market conditions change.

### 12.4 Metric-Problem Mismatch

Optimizing accuracy for an imbalanced problem, or MSE when outliers dominate, produces models that look good on paper but fail in practice. Always align the metric with the business cost of errors.

---

## 13. Choosing the Right Metric

| Problem | Recommended Metrics | Why |
|---------|-------------------|-----|
| Balanced classification | Accuracy, F1, ROC-AUC | All errors equally costly |
| Imbalanced classification | Precision, Recall, F1, PR-AUC | Accuracy is misleading |
| Medical diagnosis | Recall (sensitivity) | Missing a case is dangerous |
| Spam filtering | Precision | False alarms annoy users |
| Search ranking | Precision@K, MAP | Care about top results |
| Regression (general) | RMSE, MAE, R² | Standard error measures |
| Regression (outlier-sensitive) | MAE | Robust to extremes |
| Regression (percentage errors) | MAPE | Scale-independent |
| Probability calibration | Brier score, log loss | Predicted probabilities must be reliable |

---

## 14. The Complete Evaluation Workflow

A disciplined evaluation workflow for any supervised learning project:

1. **Split data** — train / validation / test with stratification if needed.
2. **Establish baseline** — a simple model (mean prediction, majority class, logistic regression) sets the floor.
3. **Train candidates** — try multiple algorithms with cross-validation on the training set.
4. **Tune hyperparameters** — grid or random search with cross-validation.
5. **Analyze errors** — confusion matrix, inspect misclassified examples, learning curves.
6. **Select final model** — based on validation / cross-validation performance.
7. **Evaluate once on test set** — report final metrics. Do not retrain or retune based on test results.
8. **Document** — record metrics, data splits, hyperparameters, and known limitations.

In [ ]:
# Complete workflow example
X_full, y_full = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X_full, y_full, test_size=0.2,
                                                      stratify=y_full, random_state=42)

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000))
])

cv_scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring="roc_auc")
print(f"Cross-validation ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

pipe.fit(X_train, y_train)
y_prob = pipe.predict_proba(X_test)[:, 1]
y_pred = pipe.predict(X_test)

print(f"\nFinal test set results:")
print(f"  Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_test, y_prob):.4f}")
print(f"  F1:        {f1_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=["Malignant", "Benign"]))

---

## 15. Summary

Model evaluation determines whether a machine learning system is ready for the real world. **Data splitting** (train/validation/test) and **cross-validation** provide unbiased performance estimates. **Regression metrics** (MSE, RMSE, MAE, R²) and **classification metrics** (precision, recall, F1, ROC-AUC, PR-AUC) measure different aspects of prediction quality.

The **confusion matrix** is the foundation for understanding classification errors. **Learning curves** diagnose overfitting and underfitting. **Hyperparameter tuning** with cross-validation optimizes model settings without leaking test information.

The guiding principles: never touch the test set during development, choose metrics that match the business problem, handle class imbalance explicitly, and document everything. Rigorous evaluation is what separates a model that works in a notebook from one that works in production.